# 03 · Lateness vs. review scores

**Question:** How much lower are review scores for late orders than for
on-time orders, and does the gap grow the later an order arrives?

Source: the `order_outcomes` view (`sql/02_order_outcomes.sql`): delivered
orders bought Jan 2017 - Aug 2018, latest review per order. Orders without a
review are excluded.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

# Notebooks run from notebooks/, so add src/ to the import path.
sys.path.insert(0, str(Path.cwd().parent / "src"))
from db import query

sns.set_theme(style="whitegrid")

In [ ]:
reviews = query("""
    SELECT order_id, review_score, is_late, days_late, review_before_delivery,
           customer_state, purchase_month::text AS purchase_month
    FROM order_outcomes
    WHERE review_score IS NOT NULL
""")
print(f"{len(reviews):,} reviewed orders")

## 1. Late vs. on-time

**Question:** What is the average review score for late and on-time orders,
and how sure are we of each average (95% confidence interval)?

In [ ]:
summary = (reviews.groupby("is_late")["review_score"]
           .agg(orders="count", mean_score="mean", std="std"))
summary["ci_low"] = summary["mean_score"] - 1.96 * summary["std"] / np.sqrt(summary["orders"])
summary["ci_high"] = summary["mean_score"] + 1.96 * summary["std"] / np.sqrt(summary["orders"])
summary["pct_1_star"] = reviews.groupby("is_late")["review_score"].apply(lambda s: 100 * (s == 1).mean())
summary["pct_5_star"] = reviews.groupby("is_late")["review_score"].apply(lambda s: 100 * (s == 5).mean())
summary.drop(columns="std").round(2)

**Observations:**

- 

## 2. Score distribution

**Question:** How are the 1-5 stars spread for late vs. on-time orders?
Does lateness shift every score down a little, or mainly create 1-star reviews?

In [ ]:
dist = (reviews.groupby("is_late")["review_score"]
        .value_counts(normalize=True).mul(100).rename("pct").reset_index())
dist["group"] = dist["is_late"].map({False: "On time", True: "Late"})

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=dist, x="review_score", y="pct", hue="group", ax=ax)
ax.set(title="Review score distribution", xlabel="Stars", ylabel="% of orders in group")
plt.show()
dist.pivot(index="review_score", columns="group", values="pct").round(1)

**Observations:**

- 

## 3. Score by days late

**Question:** Does the score drop gradually as an order gets later, or is
there a sharp break at the promised date? Includes early orders, to see
whether arriving *less early* already costs stars.

In [ ]:
bins   = [-np.inf, -15, -8, -1, 0, 3, 7, 14, 30, np.inf]
labels = ["15+ early", "8-14 early", "1-7 early", "on the day",
          "1-3 late", "4-7 late", "8-14 late", "15-30 late", "30+ late"]
reviews["days_late_group"] = pd.cut(reviews["days_late"], bins=bins, labels=labels)

by_days = (reviews.groupby("days_late_group", observed=True)["review_score"]
           .agg(orders="count", mean_score="mean", std="std"))
by_days["ci"] = 1.96 * by_days["std"] / np.sqrt(by_days["orders"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(by_days.index.astype(str), by_days["mean_score"], yerr=by_days["ci"],
            marker="o", capsize=4)
ax.axvline(3.5, color="black", linewidth=1)   # between "on the day" and "1-3 late"
ax.set(title="Average review score by days late (95% CI)", xlabel="", ylabel="Average stars")
plt.xticks(rotation=30)
plt.show()
by_days.drop(columns="std").round(2)

**Observations:**

- 

## 4. Adjusted for state and month

**Why adjust:** notebook 02 showed late orders are concentrated in some
**states** (Northeast) and **months** (Nov 2017, Feb-Mar 2018). If those
states or months differ in this outcome anyway, the simple comparison above
mixes that in with the effect of lateness.

The regression below compares late and on-time orders **within the same
state and purchase month**. `is_late` is the number to read: the difference
associated with being late after that adjustment. `HC1` gives robust
standard errors (no assumption that the spread is equal across groups).

This is still an **association, not proof of cause**: anything else that
differs between late and on-time orders (product type, seller, price) is
not adjusted for here.

In [ ]:
raw_gap = summary.loc[True, "mean_score"] - summary.loc[False, "mean_score"]

model = smf.ols("review_score ~ is_late + C(customer_state) + C(purchase_month)",
                data=reviews.assign(is_late=reviews["is_late"].astype(int))
               ).fit(cov_type="HC1")

adjusted = model.params["is_late"]
ci_low, ci_high = model.conf_int().loc["is_late"]
print(f"Raw gap (late - on time):      {raw_gap:+.2f} stars")
print(f"Adjusted gap (same state+month): {adjusted:+.2f} stars  (95% CI {ci_low:+.2f} to {ci_high:+.2f})")

**Observations:**

- 

## 5. Robustness: reviews written after arrival only

**Question:** Some reviews were written *before* the order arrived (often a
customer still waiting). Does the gap hold when we keep only reviews written
on or after the delivery date?

In [ ]:
after = reviews[~reviews["review_before_delivery"].astype(bool)]
print(f"Reviews written before delivery: {len(reviews) - len(after):,} "
      f"({100 * (1 - len(after) / len(reviews)):.1f}%)")

after_means = after.groupby("is_late")["review_score"].agg(orders="count", mean_score="mean")
print(after_means.round(2))
print(f"Gap using only after-arrival reviews: "
      f"{after_means.loc[True, 'mean_score'] - after_means.loc[False, 'mean_score']:+.2f} stars")

**Observations:**

- 

## Decisions from this notebook

-